# Fine-Tuning Gemma 2 2B → Apex AI (modelo PARTICULAR, portável)

Este notebook treina o **Gemma 2 2B IT** com o dataset da **Apex AI** (845+ exemplos)
e exporta um modelo **GGUF portável** que roda localmente — no Apex Desktop (.exe),
no servidor local (server.mjs) e nos apps — sem depender de provedor externo.

> ✅ **Sem token, sem login, sem lock-in.** A base do Gemma é baixada de um
> espelho aberto (Unsloth) que não exige nada. O resultado é 100% seu.

**Fluxo:**
1. Instalar dependências
2. Carregar dataset Apex (treino + teste)
3. Carregar modelo base Gemma 2 2B (aberto)
4. Configurar LoRA (fine-tuning eficiente)
5. Preparar dataset no formato instruct
6. Treinar
7. Avaliar no teste separado
8. Converter para GGUF e baixar

## 1. Instalar Dependências

Instala todas as bibliotecas necessárias para o fine-tuning.

> ⚠️ Avisos amarelos/vermelhos sobre numpy/dependências são NORMAIS no Colab e não quebram nada.

In [1]:
# ═══════════════════════════════════════════════════════════
# 1) Dependências — Gemma 4 12B precisa das libs atualizadas
# ═══════════════════════════════════════════════════════════
import subprocess, sys

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=False)

print("📦 Instalando bibliotecas (avisos são NORMAIS)...")
pip_install([
    "transformers>=4.48.0",
    "datasets",
    "accelerate>=1.5.0",
    "peft>=0.14.0",
    "bitsandbytes>=0.45.0",
    "trl>=0.15.0",
    "sentencepiece",
    "protobuf",
    "huggingface-hub",
])
print("✅ Instalação concluída.\n")

# Checagem de GPU
import torch
if torch.cuda.is_available():
    print("=" * 60)
    print(f"✅ GPU ATIVA: {torch.cuda.get_device_name(0)}")
    print(f"   Memória: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("   Pode seguir para a próxima célula. 🚀")
    print("=" * 60)
else:
    print("⚠️  SEM GPU — ative T4 em: Runtime → Change runtime type")
    print("    (Gemma 4 12B precisa de T4 ou superior)")

📦 Instalando bibliotecas (avisos são NORMAIS)...
✅ Instalação concluída.

✅ GPU ATIVA: NVIDIA RTX PRO 6000 Blackwell Server Edition
   Memória: 102.0 GB
   Pode seguir para a próxima célula. 🚀


## 2. Autenticar Hugging Face e Carregar Dataset Apex

O Gemma 4 12B é um modelo gated — você precisa de um token do Hugging Face.
Crie em: https://huggingface.co/settings/tokens

Depois, no Colab, vá em 🔑 Secrets (ícone de chave) e adicione:
- Nome: `HF_TOKEN`
- Valor: cole seu token

In [2]:
import os, json, glob, shutil

# ═══════════════════════════════════════════════
# DATASET APEX (treino + teste) — SEM TOKEN, SEM LOGIN
# ═══════════════════════════════════════════════
# Tenta baixar do GitHub. Se o repositório for PRIVADO, o download falha e o
# notebook pede o UPLOAD MANUAL dos 2 arquivos (arraste-os na janela que abrir).
REPO_RAW = "https://raw.githubusercontent.com/jedgard70/apex-ai-copilot-platform/main/training_data"

def load_jsonl(fname):
    if not os.path.exists(fname) or os.path.getsize(fname) == 0:
        return []
    with open(fname, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def remove_empty_file(fname):
    if os.path.exists(fname) and os.path.getsize(fname) == 0:
        os.remove(fname)

def normalize_uploaded_file(canonical):
    if os.path.exists(canonical) and os.path.getsize(canonical) > 0:
        return canonical
    remove_empty_file(canonical)
    prefix = canonical.replace(".jsonl", "")
    candidates = [p for p in glob.glob(prefix + "*.jsonl") if os.path.getsize(p) > 0]
    if not candidates:
        return canonical
    newest = max(candidates, key=os.path.getmtime)
    if newest != canonical:
        shutil.copyfile(newest, canonical)
        print(f"✅ Arquivo recebido como '{newest}' e normalizado para '{canonical}'")
    return canonical

print("📤 Tentando baixar o dataset da Apex do GitHub...")
os.system(f"wget -q {REPO_RAW}/apex_train.jsonl -O apex_train.jsonl")
os.system(f"wget -q {REPO_RAW}/apex_test.jsonl -O apex_test.jsonl")
remove_empty_file("apex_train.jsonl")
remove_empty_file("apex_test.jsonl")
train_raw = load_jsonl("apex_train.jsonl")
test_raw = load_jsonl("apex_test.jsonl")

# Fallback (repo privado): upload manual
if not train_raw:
    print("\n⚠️ Download automático falhou (o repositório é privado).")
    print("👉 Faça UPLOAD MANUAL agora dos 2 arquivos:")
    print("   • apex_train.jsonl")
    print("   • apex_test.jsonl")
    try:
        from google.colab import files
        files.upload()
    except Exception as e:
        print("   (Upload automático:", e, ")")
    normalize_uploaded_file("apex_train.jsonl")
    normalize_uploaded_file("apex_test.jsonl")
    train_raw = load_jsonl("apex_train.jsonl")
    test_raw = load_jsonl("apex_test.jsonl")

if not train_raw:
    print("\n⚠️  apex_train.jsonl ainda NÃO foi carregado.")
else:
    print(f"\n✅ Treino: {len(train_raw)} exemplos | Teste: {len(test_raw)} exemplos")
    print("📝 Exemplo de treino:")
    print(json.dumps(train_raw[0], ensure_ascii=False, indent=2)[:300])

📤 Tentando baixar o dataset da Apex do GitHub...

✅ Treino: 845 exemplos | Teste: 211 exemplos
📝 Exemplo de treino:
{
  "input": "O que é um RDO? Me explica rápido.",
  "output": "RDO é o Relatório Diário de Obra: registro do que aconteceu no dia — clima, efetivo, equipamentos, atividades, ocorrências e pendências. No Field Ops da Apex você monta o RDO com evidência por item."
}


## 3. Carregar o Modelo Base (Gemma 2 2B — aberto, sem token)

Usamos um espelho ABERTO do Gemma 2 2B (Unsloth) que **NÃO exige token do Hugging Face**.
Nem aceite de licença, nem login. Baixa uma vez, treina, e o resultado é 100% seu.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ══════════════════════════════════════════════════════
# MODELO BASE — Gemma 2 2B (SEM TOKEN, SEM LOGIN)
# ══════════════════════════════════════════════════════
# Usamos unsloth/gemma-2-2b-it que é um espelho ABERTO,
# NÃO precisa de token do Hugging Face.
# Nem aceite de licença, nem login.

MODEL_NAME = "unsloth/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"🔄 Carregando {MODEL_NAME} (aberto, sem token)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"\n✅ Modelo carregado!")
print(f"   {sum(p.numel() for p in model.parameters())/1e9:.2f}B parâmetros | Device: {model.device}")
print(f"   Pad token: '{tokenizer.pad_token}' id={tokenizer.pad_token_id}")

🔄 Carregando unsloth/gemma-2-2b-it (aberto, sem token)...


config.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/5.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]


✅ Modelo carregado!
   1.60B parâmetros | Device: cuda:0
   Pad token: '<pad>' id=0


In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Configuração LoRA
lora_config = LoraConfig(
    r=8,                    # Rank do LoRA (maior = mais params treináveis)
    lora_alpha=32,          # Escala (padrão para rank 8)
    target_modules=[        # Layers onde LoRA será aplicado
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Prepara modelo para treino em quantização 4-bit
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Mostra quantos parâmetros serão treinados
model.print_trainable_parameters()
# Exemplo de output: trainable params: ~8M / total: ~2.2B → só 0.4% treináveis!

trainable params: 10,383,360 || all params: 2,624,725,248 || trainable%: 0.3956


## 5. Preparar Dataset no Formato Instruct

In [5]:
from datasets import Dataset

# Formato instruct do Gemma:
# <start_of_turn>user\n{pergunta}<end_of_turn>\n<start_of_turn>model\n{resposta}<end_of_turn>
def format_example(ex):
    user_text = ex.get("input") or ex.get("messages", [{}])[0].get("content", "")
    model_text = ex.get("output") or (ex.get("messages", [{}, {}])[1].get("content", "") if ex.get("messages") else "")
    return {"text": f"<start_of_turn>user\n{user_text}<end_of_turn>\n<start_of_turn>model\n{model_text}<end_of_turn>"}

train_formatted = [format_example(ex) for ex in train_raw]
train_dataset = Dataset.from_list(train_formatted)

# Pequeno split de validação a partir do TREINO (o apex_test.jsonl fica reservado
# só para a avaliação final na célula 7 — o modelo nunca o vê durante o treino).
split = train_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"✅ Treino: {len(train_dataset)} | Validação interna: {len(eval_dataset)}")
print(f"   Teste final reservado (apex_test.jsonl): {len(test_raw)} exemplos")
print(f"\n📝 Exemplo formatado:\n{train_formatted[0]['text'][:280]}")


✅ Treino: 760 | Validação interna: 85
   Teste final reservado (apex_test.jsonl): 211 exemplos

📝 Exemplo formatado:
<start_of_turn>user
O que é um RDO? Me explica rápido.<end_of_turn>
<start_of_turn>model
RDO é o Relatório Diário de Obra: registro do que aconteceu no dia — clima, efetivo, equipamentos, atividades, ocorrências e pendências. No Field Ops da Apex você monta o RDO com evidência po


## 6. Configurar TrainingArguments e Iniciar Treinamento

In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./gemma-apex-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # Batch efetivo = 8
    learning_rate=2e-5,
    warmup_steps=10,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=50,
    save_total_limit=2,
    fp16=True,                        # Ativa mixed precision (essencial na T4)
    bf16=False,
    gradient_checkpointing=True,      # Economiza VRAM
    optim="paged_adamw_8bit",         # Otimizador eficiente para 4-bit
    report_to="none",                 # Não enviar para wandb/tensorboard
    push_to_hub=False,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
)

def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("🚀 Iniciando treinamento...")
print(f"   Batch efetivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Steps por época: ~{len(tokenized_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"   Total de steps: ~{len(tokenized_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")

trainer.train()

Map:   0%|          | 0/760 [00:00<?, ? examples/s]

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

🚀 Iniciando treinamento...
   Batch efetivo: 8
   Steps por época: ~95
   Total de steps: ~285


Step,Training Loss,Validation Loss
20,4.789755,4.710932
40,3.351610,3.216734
60,2.162404,2.400937
80,1.627271,1.757178
100,1.548297,1.395602
120,1.290210,1.211954
140,0.825806,1.065726
160,0.764657,0.961063
180,0.786991,0.882738
200,0.652657,0.820489


TrainOutput(global_step=285, training_loss=1.5882035364184464, metrics={'train_runtime': 286.7935, 'train_samples_per_second': 7.95, 'train_steps_per_second': 0.994, 'total_flos': 1.425277392519168e+16, 'train_loss': 1.5882035364184464, 'epoch': 3.0})

## 7. Avaliar no Conjunto de TESTE Separado (perguntas nunca vistas no treino)


In [7]:
import torch, random

# Salva o modelo fine-tunado (adapters LoRA)
output_dir = "./gemma-apex-finetuned-final"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Adapters LoRA salvos em {output_dir}")

def apex_generate(prompt, max_new_tokens=160):
    inputs = tokenizer(
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n",
        return_tensors="pt", truncation=True, max_length=256,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.7, do_sample=True, top_p=0.95,
                             pad_token_id=tokenizer.pad_token_id)
    resp = tokenizer.decode(out[0], skip_special_tokens=True)
    return resp.split("<start_of_turn>model\n")[-1].strip()

# ═══════════════════════════════════════════════════════════
# AVALIAÇÃO no conjunto de TESTE separado (apex_test.jsonl)
# Mede sobreposição de palavras com a resposta esperada (proxy simples de acerto).
# ═══════════════════════════════════════════════════════════
print("\n🧪 Avaliando no conjunto de teste separado...\n")
sample = test_raw if len(test_raw) <= 30 else random.sample(test_raw, 30)

def overlap_score(pred, ref):
    p = set(pred.lower().split()); r = set(ref.lower().split())
    return len(p & r) / max(1, len(r))

scores = []
for i, ex in enumerate(sample):
    q = ex.get("input", ""); expected = ex.get("output", "")
    ans = apex_generate(q)
    s = overlap_score(ans, expected); scores.append(s)
    if i < 5:  # mostra 5 exemplos
        print(f"🧑 {q}")
        print(f"🤖 {ans[:180]}")
        print(f"   overlap c/ esperado: {s:.0%}\n" + "-" * 50)

if scores:
    print(f"\n📊 Score médio de sobreposição no teste ({len(scores)} exemplos): {sum(scores)/len(scores):.1%}")
    print("   (proxy simples — quanto maior, mais o modelo aprendeu o estilo/conteúdo Apex)")


[transformers] Both `max_new_tokens` (=160) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Adapters LoRA salvos em ./gemma-apex-finetuned-final

🧪 Avaliando no conjunto de teste separado...



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧑 Pode ajudar? oi, tudo bem?
🤖 user
Pode ajudar? oi, tudo bem?
model
Oi! Tudo bem. Aonde você precisa ajudar? Posso montar orçamentos, contratos, rascunhos de contratos, análises de risco e compliance, pesquisas
   overlap c/ esperado: 9%
--------------------------------------------------


[transformers] Both `max_new_tokens` (=160) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

🧑 Quero abrir o Contracts Studio?
🤖 user
Quero abrir o Contracts Studio?
model
Basta dizer "abrir contracts studio" no chat. O Contracts Studio abre ao lado para preparar rascunho, checklist e revisão de contratos co
   overlap c/ esperado: 100%
--------------------------------------------------


[transformers] Both `max_new_tokens` (=160) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧑 O que é uma proposta comercial?
🤖 user
O que é uma proposta comercial?
model
É o documento que apresenta escopo, prazo, custo e condições de entrega para atender a solicitação do cliente. A Apex gera propostas com 
   overlap c/ esperado: 61%
--------------------------------------------------


[transformers] Both `max_new_tokens` (=160) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging


📊 Score médio de sobreposição no teste (30 exemplos): 76.8%
   (proxy simples — quanto maior, mais o modelo aprendeu o estilo/conteúdo Apex)


## 8. Exportar para GGUF Portável + Publicar no Hugging Face

Aqui o modelo vira um único arquivo **`.gguf`** que roda em qualquer lugar via
**Ollama / llama.cpp**, sem depender de nenhum provedor.

Além disso, faremos **upload do modelo para o Hugging Face Hub**
(privado ou público) para você poder baixar de qualquer máquina — PC,
servidor VPS, Oracle Cloud, ou até mesmo o Apex Desktop (.exe).

In [11]:
import os

# ═══════════════════════════════════════════════════════════
# PASSO 1: Fundir adapters LoRA no modelo base
# ═══════════════════════════════════════════════════════════
print("🔀 Fundindo adapters LoRA no modelo base...")
merged_model = model.merge_and_unload()
merged_dir = "./gemma-apex-merged"
os.makedirs(merged_dir, exist_ok=True)
merged_model.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
print(f"✅ Modelo fundido salvo em {merged_dir}/")

# ═══════════════════════════════════════════════════════════
# PASSO 2: Converter para GGUF (formato portável)
# ═══════════════════════════════════════════════════════════
print("\n🔧 Convertendo para GGUF...")

# Clona llama.cpp se necessário
if not os.path.exists("llama.cpp"):
    os.system("git clone --depth 1 https://github.com/ggerganov/llama.cpp")
os.system("pip install -q -r llama.cpp/requirements.txt gguf sentencepiece protobuf")

GGUF_FILE = "apex-ai.gguf"
size_mb = 0 # Initialize size_mb to prevent NameError if GGUF conversion fails
print("   HF → GGUF...")
os.system(f"python llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile {GGUF_FILE} --outtype q8_0")

if os.path.exists(GGUF_FILE):
    size_mb = os.path.getsize(GGUF_FILE) / 1e6
    print(f"   ✅ GGUF gerado: {GGUF_FILE} ({size_mb:.0f} MB)")
else:
    print("⚠️ Falha na conversão. Verifique os logs acima.")

# ═══════════════════════════════════════════════════════════
# PASSO 3: Guia de uso local
# ═══════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════╗
║        MODELO TREINADO COM SUCESSO!      ║
╚══════════════════════════════════════════╝

Arquivo gerado:
  {GGUF_FILE} ({size_mb:.0f} MB)

📥 Download:
  No painel do Colab (Files → refresh → {GGUF_FILE})
  clique com botão direito → Download

📦 Para usar no seu Apex Desktop (.exe):
  1. Copie {GGUF_FILE} para a pasta runtime/models/
  2. Rode o servidor:
     node server/apex-runtime/api-server.mjs
  3. Pronto! O chat vai usar seu modelo treinado.

🔧 Para usar com Ollama (opcional):
  ollama create apex-ai -f Modelfile.apex
  ollama run apex-ai "Quem é você?"
""")

🔀 Fundindo adapters LoRA no modelo base...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modelo fundido salvo em ./gemma-apex-merged/

🔧 Convertendo para GGUF...
   HF → GGUF...
⚠️ Falha na conversão. Verifique os logs acima.

╔══════════════════════════════════════════╗
║        MODELO TREINADO COM SUCESSO!      ║
╚══════════════════════════════════════════╝

Arquivo gerado:
  apex-ai.gguf (0 MB)

📥 Download:
  No painel do Colab (Files → refresh → apex-ai.gguf)
  clique com botão direito → Download

📦 Para usar no seu Apex Desktop (.exe):
  1. Copie apex-ai.gguf para a pasta runtime/models/
  2. Rode o servidor:
     node server/apex-runtime/api-server.mjs
  3. Pronto! O chat vai usar seu modelo treinado.

🔧 Para usar com Ollama (opcional):
  ollama create apex-ai -f Modelfile.apex
  ollama run apex-ai "Quem é você?"



## Bônus: Publicar no Hugging Face Hub (opcional)

Se quiser fazer backup do seu modelo na nuvem para baixar em outras máquinas.
> ⚠️ Requer token do HF com permissão de escrita.
> Crie em: https://huggingface.co/settings/tokens